## 1. Environment and configuration

In [1]:
# Imports
import os
import time
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from torchvision import models
from torchvision.models import ResNet18_Weights

from sklearn.metrics import confusion_matrix

import seaborn as sns

print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)

PyTorch: 2.14.1+cpu
Torchvision: 0.29.1+cpu


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Reproducibility and configuration
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Dataset configuration
IMAGE_SIZE = (320, 240)  # width, height
NUM_CLASSES = 29
IGNORE_INDEX = 255

BATCH_SIZE = 8
NUM_WORKERS = 2

# Training configuration
NUM_EPOCHS = 40
EARLY_STOPPING_PATIENCE = 7

ENCODER_LR = 0.0001
DECODER_LR = 0.001
WEIGHT_DECAY = 0.0001

# Loss configuration
DICE_WEIGHT = 1.0
CLASS_WEIGHT_POWER = 0.5
CLASS_WEIGHT_MIN = 0.5
CLASS_WEIGHT_MAX = 5.0

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AAI-521/project"
)

CAMVID_ROOT = PROJECT_ROOT / "datasets" / "camvid"

# Output directory
IMPROVED_MODEL_DIR = (
    CAMVID_ROOT / "models" / "improved_unet_resnet18"
)

IMPROVED_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BEST_MODEL_PATH = (
    IMPROVED_MODEL_DIR / "best_unet_resnet18.pth"
)

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Classes:", NUM_CLASSES)
print("Batch size:", BATCH_SIZE)
print("Output:", IMPROVED_MODEL_DIR)

Device: cpu
Image size: (320, 240)
Classes: 29
Batch size: 8
Output: /content/drive/MyDrive/AAI-521/project/datasets/camvid/models/improved_unet_resnet18


In [ ]:
# Reuse the validate metadata and labelling
# Preserve the 700-image sequence-based split
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

assert len(train_df) + len(val_df) + len(test_df) == 700

assert len(CLASS_TO_MODEL_ID) == NUM_CLASSES

assert SOURCE_ID_TO_MODEL_ID[31] == IGNORE_INDEX
assert SOURCE_ID_TO_MODEL_ID[32] == 11

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("Class mapping validation passed.")